In [ ]:
import warnings
from pathlib import Path
from typing import Any, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

RANDOM_STATE = 42
PLOT_DPI = 600
TOP_N_SHAP = 10
LTS_COLOR = "#80bcc8"
HTS_COLOR = "#d88f91"

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = PLOT_DPI
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


def get_code_dir() -> Path:
    """Return the script directory, or the current directory in a notebook."""
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd()


def ensure_dir(path: Path) -> None:
    """Create a directory and any missing parent directories."""
    path.mkdir(parents=True, exist_ok=True)


def save_figure(fig: plt.Figure, save_path: Path, dpi: int = PLOT_DPI) -> None:
    """Save and close a Matplotlib figure."""
    fig.savefig(
        save_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
        transparent=False,
    )
    plt.close(fig)


def build_pipeline(C: float, max_iter: int, class_weight: Optional[str]) -> Pipeline:
    """Create a standardized linear SVM pipeline."""
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "svm",
                LinearSVC(
                    C=C,
                    max_iter=max_iter,
                    class_weight=class_weight,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def load_dataset(data_path: Path) -> Tuple[pd.DataFrame, pd.Series, Sequence[str]]:
    """Load the expression matrix and validate binary labels."""
    if not data_path.exists():
        raise FileNotFoundError(f"Data file not found: {data_path}")

    df = pd.read_csv(data_path)

    if "label" not in df.columns:
        raise ValueError("The dataset must contain a 'label' column.")

    non_feature_cols = ["label"]
    first_col = df.columns[0]
    if first_col != "label" and not pd.api.types.is_numeric_dtype(df[first_col]):
        non_feature_cols.append(first_col)

    feature_cols = [column for column in df.columns if column not in non_feature_cols]
    if not feature_cols:
        raise ValueError("No feature columns were found in the dataset.")

    X_numeric = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    missing_count = int(X_numeric.isna().sum().sum())
    if missing_count > 0:
        print(
            f"[WARNING] {missing_count} missing or non-numeric feature values "
            "were replaced with 0."
        )

    X = X_numeric.fillna(0.0).astype(np.float32)
    y = pd.to_numeric(df["label"], errors="coerce")

    if y.isna().any():
        raise ValueError("The label column contains missing or non-numeric values.")

    y = y.astype(int)
    observed_labels = set(y.unique().tolist())
    if observed_labels != {0, 1}:
        raise ValueError(
            "The label column must contain both binary classes: 0 for LTS and 1 for HTS."
        )

    return X, y, feature_cols


def save_confusion_matrix(
    cm: np.ndarray,
    class_names: Sequence[str],
    save_path: Path,
) -> None:
    """Save a confusion matrix heatmap."""
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    image = ax.imshow(cm, cmap="Blues")
    fig.colorbar(image, ax=ax)

    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel="True label",
        xlabel="Predicted label",
        title="Linear SVM Test Confusion Matrix",
    )

    threshold = cm.max() / 2 if cm.max() > 0 else 0.5
    for row in range(cm.shape[0]):
        for column in range(cm.shape[1]):
            ax.text(
                column,
                row,
                int(cm[row, column]),
                ha="center",
                va="center",
                color="white" if cm[row, column] > threshold else "black",
            )

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_roc_curve(y_true: pd.Series, y_score: np.ndarray, save_path: Path) -> None:
    """Save the test-set receiver operating characteristic curve."""
    auc_value = roc_auc_score(y_true, y_score)
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, y_score)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(
        false_positive_rate,
        true_positive_rate,
        linewidth=2,
        color="black",
        label=f"ROC AUC = {auc_value:.4f}",
    )
    ax.plot([0, 1], [0, 1], "--", linewidth=1.5, color="gray")
    ax.set_title("Linear SVM Test ROC Curve")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_pr_curve(y_true: pd.Series, y_score: np.ndarray, save_path: Path) -> None:
    """Save the test-set precision-recall curve."""
    average_precision = average_precision_score(y_true, y_score)
    precision, recall, _ = precision_recall_curve(y_true, y_score)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot(recall, precision, linewidth=2, label=f"AP = {average_precision:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Linear SVM Test Precision-Recall Curve")
    ax.legend(loc="lower left")

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_score_distribution(
    y_true: pd.Series,
    y_score: np.ndarray,
    save_path: Path,
) -> None:
    """Save the distribution of test-set decision scores."""
    y_array = np.asarray(y_true)

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.hist(
        y_score[y_array == 0],
        bins=30,
        alpha=0.7,
        label="LTS (0)",
        edgecolor="black",
        color=LTS_COLOR,
    )
    ax.hist(
        y_score[y_array == 1],
        bins=30,
        alpha=0.7,
        label="HTS (1)",
        edgecolor="black",
        color=HTS_COLOR,
    )
    ax.set_xlabel("Decision score toward HTS")
    ax.set_ylabel("Cell count")
    ax.set_title("Linear SVM Test Decision Score Distribution")
    ax.legend()

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_top_coefficients(
    coefficient_df: pd.DataFrame,
    save_path: Path,
    top_n: int = 30,
) -> None:
    """Plot genes with the largest absolute final-model coefficients."""
    plot_df = (
        coefficient_df.sort_values("abs_coefficient", ascending=False)
        .head(top_n)
        .sort_values("coefficient", ascending=True)
    )

    colors = [HTS_COLOR if value > 0 else LTS_COLOR for value in plot_df["coefficient"]]

    fig, ax = plt.subplots(figsize=(9, max(7, top_n * 0.32)))
    ax.barh(
        plot_df["gene"],
        plot_df["coefficient"],
        color=colors,
        edgecolor="black",
        linewidth=0.6,
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Final linear SVM coefficient")
    ax.set_ylabel("Gene")
    ax.set_title(
        f"Top {len(plot_df)} Genes by Absolute Linear SVM Coefficient\n"
        "Positive = HTS association; Negative = LTS association"
    )

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_top_shap_importance(
    shap_df: pd.DataFrame,
    save_path: Path,
    top_n: int = TOP_N_SHAP,
) -> None:
    """Plot genes ranked by mean absolute SHAP value."""
    plot_df = (
        shap_df.sort_values("mean_abs_shap", ascending=False)
        .head(top_n)
        .sort_values("mean_abs_shap", ascending=True)
    )

    fig, ax = plt.subplots(figsize=(8.5, max(5.5, top_n * 0.48)))
    scatter = ax.scatter(
        plot_df["mean_abs_shap"],
        plot_df["gene"],
        c=plot_df["mean_abs_shap"],
        cmap="RdYlBu_r",
        s=100,
        edgecolors="black",
        linewidths=0.3,
    )
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_ylabel("Gene")
    ax.set_title(f"Top {len(plot_df)} Gene Importance by Final Linear SVM SHAP")
    fig.colorbar(scatter, ax=ax, label="Importance")

    fig.tight_layout()
    save_figure(fig, save_path)


def plot_decision_boundary_pca(
    best_params: dict,
    X_trainval: pd.DataFrame,
    y_trainval: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    save_path: Path,
) -> None:
    """Create an illustrative two-dimensional PCA decision-boundary plot."""
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_trainval)
    X_test_scaled = scaler.transform(X_test)

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    visualization_model = LinearSVC(
        C=best_params["svm__C"],
        max_iter=best_params["svm__max_iter"],
        class_weight=best_params["svm__class_weight"],
        random_state=RANDOM_STATE,
    )
    visualization_model.fit(X_train_pca, y_trainval)

    x_min = X_train_pca[:, 0].min() - 1.0
    x_max = X_train_pca[:, 0].max() + 1.0
    y_min = X_train_pca[:, 1].min() - 1.0
    y_max = X_train_pca[:, 1].max() + 1.0

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400),
    )
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    decision_surface = visualization_model.decision_function(grid_points).reshape(xx.shape)

    y_train_array = np.asarray(y_trainval)
    y_test_array = np.asarray(y_test)

    fig, ax = plt.subplots(figsize=(8, 6.8))
    ax.contourf(xx, yy, decision_surface > 0, alpha=0.22, cmap="coolwarm")
    ax.contour(
        xx,
        yy,
        decision_surface,
        levels=[-1, 0, 1],
        linestyles=["--", "-", "--"],
        linewidths=[1.2, 2.0, 1.2],
        colors="black",
    )

    ax.scatter(
        X_train_pca[y_train_array == 0, 0],
        X_train_pca[y_train_array == 0, 1],
        label="Training+Validation LTS",
        alpha=0.75,
        edgecolors="black",
        linewidths=0.3,
        s=45,
        color=LTS_COLOR,
    )
    ax.scatter(
        X_train_pca[y_train_array == 1, 0],
        X_train_pca[y_train_array == 1, 1],
        label="Training+Validation HTS",
        alpha=0.75,
        edgecolors="black",
        linewidths=0.3,
        s=45,
        color=HTS_COLOR,
    )
    ax.scatter(
        X_test_pca[y_test_array == 0, 0],
        X_test_pca[y_test_array == 0, 1],
        label="Test LTS",
        marker="s",
        edgecolors="black",
        linewidths=0.6,
        s=55,
        color=LTS_COLOR,
    )
    ax.scatter(
        X_test_pca[y_test_array == 1, 0],
        X_test_pca[y_test_array == 1, 1],
        label="Test HTS",
        marker="^",
        edgecolors="black",
        linewidths=0.6,
        s=60,
        color=HTS_COLOR,
    )

    training_scores = visualization_model.decision_function(X_train_pca)
    margin_mask = np.abs(training_scores) <= 1.05
    ax.scatter(
        X_train_pca[margin_mask, 0],
        X_train_pca[margin_mask, 1],
        facecolors="none",
        edgecolors="black",
        s=120,
        linewidths=1.0,
        label="Near margin",
    )

    explained_variance = pca.explained_variance_ratio_ * 100
    ax.set_xlabel(f"PC1 ({explained_variance[0]:.2f}% variance)")
    ax.set_ylabel(f"PC2 ({explained_variance[1]:.2f}% variance)")
    ax.set_title("Illustrative Linear SVM Decision Boundary in Two-Dimensional PCA Space")
    ax.legend(loc="best", fontsize=9)

    fig.tight_layout()
    save_figure(fig, save_path)


def get_linear_shap_values(
    explainer: Any,
    X_scaled: np.ndarray,
) -> np.ndarray:
    """Extract a two-dimensional SHAP array across SHAP versions."""
    try:
        explanation = explainer(X_scaled)
        shap_values = np.asarray(explanation.values)
    except Exception:
        shap_values = np.asarray(explainer.shap_values(X_scaled))

    if shap_values.ndim == 3 and shap_values.shape[-1] == 2:
        shap_values = shap_values[:, :, 1]

    if shap_values.ndim != 2:
        raise ValueError(f"Unexpected SHAP output shape: {shap_values.shape}")

    return shap_values


def save_shap_summary_plot(
    shap_values: np.ndarray,
    X_scaled: np.ndarray,
    feature_cols: Sequence[str],
    save_path: Path,
) -> None:
    """Save a SHAP summary beeswarm plot."""
    import shap

    shap.summary_plot(
        shap_values,
        X_scaled,
        feature_names=feature_cols,
        show=False,
    )
    fig = plt.gcf()
    fig.set_size_inches(10, 8)
    save_figure(fig, save_path)


def tune_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    output_path: Path,
) -> Tuple[dict, float]:
    """Select hyperparameters by validation-set ROC AUC."""
    parameter_grid = {
        "svm__C": [0.01, 0.1, 1, 10],
        "svm__max_iter": [3000, 5000],
        "svm__class_weight": ["balanced", None],
    }

    best_validation_auc = -np.inf
    best_params = None
    tuning_records = []

    print("[INFO] Tuning linear SVM hyperparameters on the validation set...")

    for params in ParameterGrid(parameter_grid):
        pipeline = build_pipeline(
            C=params["svm__C"],
            max_iter=params["svm__max_iter"],
            class_weight=params["svm__class_weight"],
        )

        with warnings.catch_warnings(record=True) as caught_warnings:
            warnings.simplefilter("always", ConvergenceWarning)
            pipeline.fit(X_train, y_train)

        validation_score = pipeline.decision_function(X_val)
        validation_prediction = (validation_score > 0).astype(int)

        validation_auc = roc_auc_score(y_val, validation_score)
        validation_accuracy = accuracy_score(y_val, validation_prediction)
        validation_f1 = f1_score(y_val, validation_prediction, zero_division=0)
        validation_precision = precision_score(
            y_val, validation_prediction, zero_division=0
        )
        validation_recall = recall_score(y_val, validation_prediction, zero_division=0)

        convergence_warning = any(
            issubclass(warning.category, ConvergenceWarning)
            for warning in caught_warnings
        )

        tuning_records.append(
            {
                "C": params["svm__C"],
                "max_iter": params["svm__max_iter"],
                "class_weight": params["svm__class_weight"],
                "validation_auc": validation_auc,
                "validation_accuracy": validation_accuracy,
                "validation_f1": validation_f1,
                "validation_precision": validation_precision,
                "validation_recall": validation_recall,
                "convergence_warning": convergence_warning,
            }
        )

        print(
            f"[INFO] C={params['svm__C']:<5} "
            f"max_iter={params['svm__max_iter']:<5} "
            f"class_weight={str(params['svm__class_weight']):<8} "
            f"Validation AUC={validation_auc:.4f} "
            f"Validation ACC={validation_accuracy:.4f} "
            f"Validation F1={validation_f1:.4f}"
        )

        if validation_auc > best_validation_auc:
            best_validation_auc = validation_auc
            best_params = params.copy()

    if best_params is None:
        raise RuntimeError("No linear SVM model was trained successfully.")

    tuning_df = pd.DataFrame(tuning_records).sort_values(
        "validation_auc", ascending=False
    )
    tuning_df.to_csv(output_path, index=False)

    return best_params, best_validation_auc


def run_final_model_shap(
    final_pipeline: Pipeline,
    X_trainval: pd.DataFrame,
    X_all: pd.DataFrame,
    feature_cols: Sequence[str],
    out_dir: Path,
    fig_dir: Path,
) -> None:
    """Explain the refitted final model and summarize SHAP values over all cells."""
    try:
        import shap

        print("[INFO] Calculating SHAP values for the refitted final linear SVM model...")

        scaler = final_pipeline.named_steps["scaler"]
        svm_model = final_pipeline.named_steps["svm"]

        X_trainval_scaled = scaler.transform(X_trainval)
        X_all_scaled = scaler.transform(X_all)

        explainer = shap.LinearExplainer(svm_model, X_trainval_scaled)
        shap_values = get_linear_shap_values(explainer, X_all_scaled)

        save_shap_summary_plot(
            shap_values,
            X_all_scaled,
            feature_cols,
            fig_dir / "final_linear_svm_shap_summary.png",
        )

        coefficients = svm_model.coef_.ravel()
        shap_df = pd.DataFrame(
            {
                "gene": feature_cols,
                "mean_abs_shap": np.abs(shap_values).mean(axis=0),
                "mean_shap": shap_values.mean(axis=0),
                "coefficient": coefficients,
                "abs_coefficient": np.abs(coefficients),
                "association": np.where(coefficients > 0, "HTS", "LTS"),
            }
        ).sort_values("mean_abs_shap", ascending=False)

        shap_df.to_csv(
            out_dir / "final_linear_svm_shap_gene_importance.csv",
            index=False,
        )
        shap_df.head(TOP_N_SHAP).to_csv(
            out_dir / f"top{TOP_N_SHAP}_final_linear_svm_shap_genes.csv",
            index=False,
        )

        hts_df = shap_df[shap_df["coefficient"] > 0].sort_values(
            "mean_abs_shap", ascending=False
        )
        lts_df = shap_df[shap_df["coefficient"] < 0].sort_values(
            "mean_abs_shap", ascending=False
        )
        hts_df.head(TOP_N_SHAP).to_csv(
            out_dir / f"top{TOP_N_SHAP}_hts_associated_genes.csv",
            index=False,
        )
        lts_df.head(TOP_N_SHAP).to_csv(
            out_dir / f"top{TOP_N_SHAP}_lts_associated_genes.csv",
            index=False,
        )

        plot_top_shap_importance(
            shap_df,
            fig_dir / f"top{TOP_N_SHAP}_final_linear_svm_shap_importance.png",
            top_n=TOP_N_SHAP,
        )

        print("[INFO] Final-model SHAP analysis completed successfully.")

    except Exception as error:
        raise RuntimeError(f"Final-model SHAP analysis failed: {error}") from error


def main() -> None:
    code_dir = get_code_dir()
    data_path = code_dir / "df_expr.csv"
    out_dir = code_dir / "linear_svm_hts_lts_results"
    fig_dir = out_dir / "figures"

    ensure_dir(out_dir)
    ensure_dir(fig_dir)

    print(f"[INFO] Working directory: {code_dir}")
    print(f"[INFO] Reading dataset: {data_path}")

    X, y, feature_cols = load_dataset(data_path)

    print(f"[INFO] Feature matrix shape: {X.shape}")
    print(f"[INFO] HTS cells (label 1): {int((y == 1).sum())}")
    print(f"[INFO] LTS cells (label 0): {int((y == 0).sum())}")

    X_temp, X_test, y_temp, y_test = train_test_split(
        X,
        y,
        test_size=1 / 3,
        stratify=y,
        random_state=RANDOM_STATE,
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        stratify=y_temp,
        random_state=RANDOM_STATE,
    )

    print(f"[INFO] Training-set size: {X_train.shape[0]}")
    print(f"[INFO] Validation-set size: {X_val.shape[0]}")
    print(f"[INFO] Test-set size: {X_test.shape[0]}")

    best_params, best_validation_auc = tune_model(
        X_train,
        y_train,
        X_val,
        y_val,
        out_dir / "validation_tuning_results.csv",
    )

    print(f"[INFO] Best parameters: {best_params}")
    print(f"[INFO] Best validation ROC AUC: {best_validation_auc:.4f}")

    X_trainval = pd.concat([X_train, X_val], axis=0)
    y_trainval = pd.concat([y_train, y_val], axis=0)

    final_pipeline = build_pipeline(
        C=best_params["svm__C"],
        max_iter=best_params["svm__max_iter"],
        class_weight=best_params["svm__class_weight"],
    )

    print("[INFO] Refitting the final linear SVM model on training and validation data...")
    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always", ConvergenceWarning)
        final_pipeline.fit(X_trainval, y_trainval)

    final_convergence_warning = any(
        issubclass(warning.category, ConvergenceWarning)
        for warning in caught_warnings
    )
    if final_convergence_warning:
        print("[WARNING] The refitted final model produced a convergence warning.")

    coefficients = final_pipeline.named_steps["svm"].coef_.ravel()
    coefficient_df = pd.DataFrame(
        {
            "gene": feature_cols,
            "coefficient": coefficients,
            "abs_coefficient": np.abs(coefficients),
            "association": np.where(coefficients > 0, "HTS", "LTS"),
        }
    ).sort_values("abs_coefficient", ascending=False)
    coefficient_df.to_csv(
        out_dir / "final_linear_svm_coefficients.csv",
        index=False,
    )

    print("[INFO] Evaluating the final model on the independent test set...")
    test_score = final_pipeline.decision_function(X_test)
    test_prediction = (test_score > 0).astype(int)

    test_auc = roc_auc_score(y_test, test_score)
    test_accuracy = accuracy_score(y_test, test_prediction)
    test_f1 = f1_score(y_test, test_prediction, zero_division=0)
    test_precision = precision_score(y_test, test_prediction, zero_division=0)
    test_recall = recall_score(y_test, test_prediction, zero_division=0)
    test_average_precision = average_precision_score(y_test, test_score)

    metrics_df = pd.DataFrame(
        [
            {
                "best_validation_auc": best_validation_auc,
                "test_auc": test_auc,
                "test_accuracy": test_accuracy,
                "test_f1": test_f1,
                "test_precision": test_precision,
                "test_recall": test_recall,
                "test_average_precision": test_average_precision,
                "best_C": best_params["svm__C"],
                "best_max_iter": best_params["svm__max_iter"],
                "best_class_weight": best_params["svm__class_weight"],
                "final_convergence_warning": final_convergence_warning,
            }
        ]
    )
    metrics_df.to_csv(out_dir / "test_metrics.csv", index=False)

    report = classification_report(
        y_test,
        test_prediction,
        target_names=["LTS", "HTS"],
        digits=4,
        zero_division=0,
    )
    with open(out_dir / "classification_report.txt", "w", encoding="utf-8") as file:
        file.write(report)

    test_predictions_df = pd.DataFrame(
        {
            "true_label": y_test.to_numpy(),
            "true_class": y_test.map({0: "LTS", 1: "HTS"}).to_numpy(),
            "predicted_label": test_prediction,
            "predicted_class": pd.Series(test_prediction).map({0: "LTS", 1: "HTS"}),
            "decision_score_toward_HTS": test_score,
        }
    )
    test_predictions_df.to_csv(out_dir / "test_predictions.csv", index=False)

    confusion = confusion_matrix(y_test, test_prediction)
    save_confusion_matrix(
        confusion,
        ["LTS", "HTS"],
        fig_dir / "test_confusion_matrix.png",
    )
    plot_roc_curve(y_test, test_score, fig_dir / "test_roc_curve.png")
    plot_pr_curve(y_test, test_score, fig_dir / "test_pr_curve.png")
    plot_score_distribution(
        y_test,
        test_score,
        fig_dir / "test_decision_score_distribution.png",
    )
    plot_top_coefficients(
        coefficient_df,
        fig_dir / "top30_final_linear_svm_coefficients.png",
        top_n=30,
    )
    plot_decision_boundary_pca(
        best_params,
        X_trainval,
        y_trainval,
        X_test,
        y_test,
        fig_dir / "illustrative_decision_boundary_pca.png",
    )

    run_final_model_shap(
        final_pipeline,
        X_trainval,
        X,
        feature_cols,
        out_dir,
        fig_dir,
    )

    print("[RESULT] Analysis completed successfully.")
    print(f"[RESULT] Output directory: {out_dir}")
    print(f"[RESULT] Best parameters: {best_params}")
    print(f"[RESULT] Best validation ROC AUC: {best_validation_auc:.4f}")
    print(f"[RESULT] Test ROC AUC: {test_auc:.4f}")
    print(f"[RESULT] Test accuracy: {test_accuracy:.4f}")
    print(f"[RESULT] Test F1 score: {test_f1:.4f}")
    print(f"[RESULT] Test precision: {test_precision:.4f}")
    print(f"[RESULT] Test recall: {test_recall:.4f}")
    print(f"[RESULT] Test average precision: {test_average_precision:.4f}")


if __name__ == "__main__":
    main()
